# Notebook 3 · Organização e exploração do resultado integrado

**CIV2155 · FNDE Tipo 2 R03 · Continuidade do Notebook 2.**

As funções Python, o perfil próprio, as restrições SHACL e as oito consultas estão nas células, com explicação, fontes e limites por trecho. O esquema público BOT é uma referência externa versionada, carregada como dado e conferida por hash. Não há importação de módulos Python do projeto.

O procedimento conserva os cinco grafos do N2 e acrescenta organização, contexto de propriedades da V137, validação e um índice para exploração. A geometria não é recalculada. No Colab, usa as pastas oficiais `.execucao/notebook3` e `resultados/organizacao` do projeto no Drive. Execute as células em ordem.

O resultado de referência do N2 contém 415.175 quádruplas e 26.601 produtos. Uma nova execução do N2 pode gerar um ZIP de bytes diferentes. Por isso, este notebook verifica os cinco grafos e as tabelas de apoio contra a referência semântica declarada, além de registrar o hash do arquivo que efetivamente recebeu. Resultados incompatíveis são recusados.


## 1. Preparar as entradas

### 1.1. Conectar o Drive e definir destinos

**O que este trecho faz:** Confere a pasta do projeto, prepara os destinos e define o download de entradas com versão fixada.

**Referências deste trecho:** [Google Colab e Drive](https://research.google.com/colaboratory/faq.html); [Python 3.12](https://docs.python.org/3.12/library/).

**Aplicação no projeto:** Mesmo padrão de pastas do N1 e N2. Os caminhos exibidos são relativos e os avisos conservam a mensagem da biblioteca.

**Limite:** A autorização do Drive é feita pelo Google. A preparação das pastas não executa a transformação.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, time
from urllib.request import urlretrieve
from importlib.metadata import version


VERSAO_ENTRADAS = "a919e7d10dcfd1269737379c166202948d93c569"
URL_ENTRADAS = (
    f"https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/{VERSAO_ENTRADAS}/"
)
PASTA_PROJETO_DRIVE = "bim-ontology-CIV2155"


def configurar_pastas(raiz_local=None, *, em_colab=None, montagem="/content/drive"):
    if em_colab is None:
        em_colab = "google.colab" in sys.modules
    if em_colab:
        from google.colab import drive

        drive.mount(str(montagem))
        raiz = Path(montagem) / "MyDrive" / PASTA_PROJETO_DRIVE
        if not raiz.is_dir():
            raise FileNotFoundError(
                "A pasta bim-ontology-CIV2155 não foi encontrada em Meu Drive. "
                "Confira a conta conectada antes de continuar."
            )
    else:
        raiz = Path(raiz_local or Path.cwd() / "CIV2155")
    raiz = raiz.resolve()
    trabalho, saida = raiz / ".execucao", raiz / "resultados"
    for pasta in [raiz, trabalho, saida]:
        pasta.mkdir(parents=True, exist_ok=True)
    return raiz, trabalho, saida


RAIZ, TRABALHO, SAIDA = configurar_pastas(os.environ.get("CIV2155_RAIZ"))
TRABALHO = TRABALHO / "notebook3"
SAIDA = SAIDA / "organizacao"
TRABALHO.mkdir(parents=True, exist_ok=True)
SAIDA.mkdir(parents=True, exist_ok=True)
INICIO_UTC = datetime.now(timezone.utc).isoformat()


def caminho_no_projeto(p):
    return Path(p).resolve().relative_to(RAIZ).as_posix()


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def obter_entrada(relativo, esperado):
    relativo = Path(relativo)
    destino = (
        TRABALHO / "entradas" / relativo.name
        if relativo.parts[0] in ("dados", "referencias")
        else RAIZ / relativo
    )
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        urlretrieve(URL_ENTRADAS + relativo.as_posix(), destino)
    if sha(destino) != esperado:
        raise ValueError("Entrada diferente da versão registrada: " + relativo.as_posix())
    return destino


print("Pasta de trabalho no projeto:", caminho_no_projeto(TRABALHO))
print("Pasta de resultados no projeto:", caminho_no_projeto(SAIDA))

# O nome do arquivo da biblioteca basta para localizar um aviso, sem expor o computador.
import warnings

_formatar_aviso_original = warnings.formatwarning


def aviso_portatil(message, category, filename, lineno, line=None):
    return _formatar_aviso_original(message, category, Path(filename).name, lineno, line)


warnings.formatwarning = aviso_portatil


def localizar_entrada(relativo):
    return TRABALHO / "entradas" / Path(relativo).name


Pasta de trabalho no projeto: .execucao/notebook3
Pasta de resultados no projeto: resultados/organizacao


### 1.2. Bibliotecas usadas nesta etapa

**O que este trecho faz:** Confere RDFLib, IfcOpenShell e pySHACL nas versões do estudo.

**Referências deste trecho:** [RDFLib 7.1.4](https://rdflib.readthedocs.io/en/7.1.4/); [IfcOpenShell, geometria](https://docs.ifcopenshell.org/ifcopenshell-python/geometry_processing.html); [pySHACL](https://github.com/RDFLib/pySHACL).

**Aplicação no projeto:** O N3 usa o Dataset já integrado e lê propriedades IFC. Não depende de Java nem de novo cálculo geométrico.

**Limite:** A conformidade depende do contrato e da projeção explicitados adiante.


In [2]:
dependencias = {
    "rdflib": "rdflib==7.1.4",
    "ifcopenshell": "ifcopenshell==0.8.4",
    "pyshacl": "pyshacl==0.30.1",
}
instalar = [
    spec
    for mod, spec in dependencias.items()
    if importlib.util.find_spec(mod) is None or version(mod) != spec.split("==")[1]
]
if instalar:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *instalar])
VERSOES_EXECUCAO = {mod: version(mod) for mod in dependencias}
from IPython.display import display

print(VERSOES_EXECUCAO)


{'rdflib': '7.1.4', 'ifcopenshell': '0.8.4', 'pyshacl': '0.30.1'}


### 1.3. Dataset do N2, IFC SCO e esquema BOT

**O que este trecho faz:** Prepara o resultado existente do N2 ou obtém a referência publicada quando o arquivo está ausente. Confere o IFC e o esquema externo.

**Referências deste trecho:** [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [Rasmussen et al. (2021), §3.4](https://doi.org/10.3233/SW-200385); [FNDE, Projeto Tipo 2](https://www.gov.br/fnde/pt-br/acesso-a-informacao/acoes-e-programas/programas/proinfancia/projetos-arquitetonicos-para-construcao/projeto-tipo-2).

**Aplicação no projeto:** Não substitui um resultado local do N2. A aceitação é decidida pela verificação semântica da etapa 2. O esquema BOT 0.3.2 conserva integralmente autoria, versão e licença CC BY 1.0.

**Limite:** O esquema BOT é dado de referência externo. A presença de uma hierarquia no esquema não implica execução de raciocinador OWL.


In [3]:
CONFIG_SCO = {
    "arquivo": "TIPO2-SCO-MOD_R03.IFC",
    "zip": "dados/TIPO2-SCO-MOD_R03.IFC.zip",
    "sha256_zip": "9f9f0574df57606690d85cf1c875b067dd6df21e2e28a2d65bc53bfa26408efe",
    "sha256_ifc": "bde759756938510c56112b9544aa2a26aca08b6a229008de85c5144d2961ea93",
    "fator_comprimento_para_m": 0.01,
    "namespace": "https://github.com/prcgui/bim-ontology-CIV2155/id/estrutura-sco/",
}
ENTRADAS_PUBLICADAS = {
    "resultados/integracao/Integracao_Arquitetura_Estrutura.zip": "8fb829827ec1ad3b02285b0f24d8d8329c7c999bf3904c50533c37e3294f5959",
    "resultados/integracao/Identidades_Modelos.csv": "cac55b07044cb41814e382b41f2135b7dd1ee8ae3a6a14a21393022e36af053a",
    "resultados/integracao/Integracao_Por_Ambiente.csv": "1b9355099fcbf56280abbca0aea7cfef8813e68375dd27a0771bf7955837e49e",
}
for relativo, digest in ENTRADAS_PUBLICADAS.items():
    destino = RAIZ / relativo
    if not destino.exists():
        obter_entrada(relativo, digest)
        print("Referência publicada obtida:", destino.name)
    else:
        print("Entrada existente, a conferir semanticamente:", destino.name)
obter_entrada(CONFIG_SCO["zip"], CONFIG_SCO["sha256_zip"])
ESQUEMA_BOT = obter_entrada(
    "referencias/organizacao/BOT_0.3.2.ttl",
    "0f51c7f192d5e694a9c465cbc883448c64790ffabc3c1e8d6530912c45510d73",
)


Entrada existente, a conferir semanticamente: Integracao_Arquitetura_Estrutura.zip
Entrada existente, a conferir semanticamente: Identidades_Modelos.csv
Entrada existente, a conferir semanticamente: Integracao_Por_Ambiente.csv


## 2. Conferir a base integrada

### 2.1. Identificadores e assinaturas da referência

**O que este trecho faz:** Expõe namespaces, grafos, hash de referência e assinaturas usadas para conferir entradas recém-produzidas.

**Referências deste trecho:** [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [W3C PROV-O](https://www.w3.org/TR/prov-o/); [Ziegler e Dittrich (2007)](https://doi.org/10.1007/978-3-540-72677-7_3); [Python 3.12](https://docs.python.org/3.12/library/).

**Aplicação no projeto:** As assinaturas por grafo e as assinaturas canônicas das tabelas foram calculadas sobre os artefatos publicados da versão a919e7d. A ordem das linhas CSV e a embalagem ZIP não mudam sua aceitação.

**Limite:** Este estudo exige equivalência com a referência delimitada. Um novo resultado com conteúdo semântico diferente deve ser investigado e a referência revisada explicitamente.


In [4]:
"""Organização incremental do N2, sem nova inferência geométrica.

Bonduel et al. (2018), §§1.3 e 2.1: recursos e contexto de propriedades.
buildingSMART IFC4 IfcPropertySingleValue: unidade explícita ou unidade de projeto
correspondente ao tipo da medida. Não elevar a unidade linear ao quadrado/cubo.
W3C SHACL (2017) e PROV-O (2013): contrato próprio e proveniência de execução.
O perfil não implementa gUFO, gOCED, GoDa nem harmonização global de propriedades.
"""

from pathlib import Path

from collections import Counter

from datetime import datetime, timezone

import csv, json, zipfile, hashlib, time

from rdflib import Dataset, Graph, Namespace, URIRef, Literal, RDF, RDFS, XSD, BNode

from rdflib.compare import to_isomorphic

BASE = "https://github.com/prcgui/bim-ontology-CIV2155/"

C = Namespace(BASE + "vocab/analise#")

N = Namespace(BASE + "vocab/integracao#")

CTX = Namespace(BASE + "vocab/contexto#")

B = Namespace(BASE + "id/integracao/")

BOT = Namespace("https://w3id.org/bot#")

PROV = Namespace("http://www.w3.org/ns/prov#")

P = Namespace("http://lbd.arch.rwth-aachen.de/props#")

GRAPHS = {
    k: URIRef(BASE + "grafo/" + k)
    for k in [
        "arquitetura",
        "smt",
        "sco",
        "relacoes",
        "proveniencia",
        "perfil",
        "bot-referencia",
        "propriedades-contextuais",
        "execucao-n3",
    ]
}

V137 = "0EVSuIeKOHxuDHo7zKqQBn"

EXPECTED_SOURCE = "8fb829827ec1ad3b02285b0f24d8d8329c7c999bf3904c50533c37e3294f5959"

sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

REFERENCIA_GRAFOS_N2 = {
    "arquitetura": "4642507359555821943394177105577707749316735264117634904829773727968520781487809391",
    "smt": "4894872651121198898811690104796682575294517874901550405310861576115937635996647893",
    "sco": "11882447438810760627625410030609227949547383327150759063203197044046537188165137416",
    "relacoes": "1058679977442546220032696355506674636639788331366913650487038008737720348802249072",
    "proveniencia": "1531714350577164184474852220496309583508325115454230596758639693056683508458951925",
}
REFERENCIA_TABELAS_N2 = {
    "Identidades_Modelos.csv": "ffcf92aa587506062f084a9f5c3f93c74a6b73dab64b7a2237f353661cd4bcb6",
    "Integracao_Por_Ambiente.csv": "51d9e46ffd89f7ac13382cef6d1a8331b6be09c7333674e18c59daf500ad4bd9",
}


def salvar(p, obj, compacto=False):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=None if compacto else 2,
            separators=(",", ":") if compacto else None,
        ),
        encoding="utf8",
    )


def assinatura_tabela(path):
    with Path(path).open(encoding="utf-8-sig") as fp:
        rows = list(csv.DictReader(fp))
    canon = sorted(json.dumps(row, sort_keys=True, ensure_ascii=False) for row in rows)
    return hashlib.sha256("\n".join(canon).encode()).hexdigest()


### 2.2. Carregar e conferir o N2

**O que este trecho faz:** Lê o N-Quads, confere as tabelas, produtos e os cinco grafos, e registra o hash real do ZIP recebido.

**Referências deste trecho:** [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [W3C N-Quads](https://www.w3.org/TR/n-quads/); [RDFLib 7.1.4](https://rdflib.readthedocs.io/en/7.1.4/).

**Aplicação no projeto:** A verificação aceita uma nova embalagem ZIP que preserve o conteúdo semântico e as tabelas. A referência não é substituída silenciosamente.

**Limite:** A compatibilidade aqui é informacional; não é uma nova validação geométrica ou de projeto.


In [5]:
def carregar(raiz):
    global EXPECTED_SOURCE
    raiz = Path(raiz)
    source = raiz / "resultados/integracao/Integracao_Arquitetura_Estrutura.zip"
    EXPECTED_SOURCE = sha(source)
    for nome, digest in REFERENCIA_TABELAS_N2.items():
        if assinatura_tabela(raiz / "resultados/integracao" / nome) != digest:
            raise ValueError("Tabela N2 diferente da referência semântica: " + nome)
    ds = Dataset(default_union=True)
    with zipfile.ZipFile(source) as z:
        ds.parse(data=z.read("Integracao_Arquitetura_Estrutura.nq"), format="nquads")
    identities = list(
        csv.DictReader(
            (raiz / "resultados/integracao/Identidades_Modelos.csv").open(encoding="utf-8-sig")
        )
    )
    signatures = {
        k: str(to_isomorphic(ds.graph(GRAPHS[k])).graph_digest()) for k in list(GRAPHS)[:5]
    }
    if len(identities) != 26601 or sum(len(ds.graph(GRAPHS[k])) for k in signatures) != 415175:
        raise ValueError("Contagens de entrada inesperadas.")
    if signatures != REFERENCIA_GRAFOS_N2:
        raise ValueError("Os cinco grafos do N2 diferem da referência semântica declarada.")
    print("Hash do ZIP N2 efetivamente recebido:", EXPECTED_SOURCE)
    return ds, identities, signatures


### 2.3. Executar a leitura e identificar a V137

**O que este trecho faz:** Carrega a entrada e mostra produto, GUID, tipos e proveniência da viga usada como exemplo.

**Referências deste trecho:** [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [W3C PROV-O](https://www.w3.org/TR/prov-o/); [Rasmussen et al. (2021), §3.4](https://doi.org/10.3233/SW-200385).

**Aplicação no projeto:** Distingue o indivíduo descrito, sua classe IFC e seus tipos RDF. Os modelos representam arquivos de origem.

**Limite:** Nomes iguais não unem recursos. A ausência de axiomas no N2 não significa ausência de um esquema BOT externo.


In [6]:
dataset, identidades, assinaturas_n2 = carregar(RAIZ)
print("Produtos preservados:", len(identidades))
print("Quádruplas N2:", sum(len(dataset.graph(GRAPHS[k])) for k in assinaturas_n2))
print(
    "Hierarquias declaradas no N2:", len(list(dataset.triples((None, RDFS.subClassOf, None))))
)
v137 = next(r for r in identidades if r["modelo"] == "sco" and r["guid"] == V137)
display(v137)
for p in [RDF.type, P.globalIdIfcRoot_attribute_simple, PROV.wasDerivedFrom]:
    for value in dataset.objects(URIRef(v137["uri"]), p):
        print(p.n3(), value.n3())


Hash do ZIP N2 efetivamente recebido: 1df6f71ae28bc7726d70226b3cbb2eb8df2538b298acd5cacf409a3d379e094b
Produtos preservados: 26601
Quádruplas N2: 415175
Hierarquias declaradas no N2: 0


{'modelo': 'sco',
 'guid': '0EVSuIeKOHxuDHo7zKqQBn',
 'classe_ifc': 'IfcBeam',
 'nome': 'V137',
 'uri': 'https://github.com/prcgui/bim-ontology-CIV2155/id/estrutura-sco/beam_0e7dce12-a146-11ef-8351-c87f54d1a2f1',
 'pavimento_ifc': '37aq1vEsn4dQ0oyWcjKKws'}

<http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <https://w3id.org/bot#Element>
<http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <https://pi.pauwel.be/voc/buildingelement#Beam>
<http://lbd.arch.rwth-aachen.de/props#globalIdIfcRoot_attribute_simple> "0EVSuIeKOHxuDHo7zKqQBn"
<http://www.w3.org/ns/prov#wasDerivedFrom> <https://github.com/prcgui/bim-ontology-CIV2155/id/integracao/modelo/sco>


## 3. Explicitar o perfil de organização

### 3.1. Perfil informacional do estudo

**O que este trecho faz:** Expõe integralmente as classes e propriedades acrescentadas para distinguir registros de resultados e recursos descritos.

**Referências deste trecho:** [Ziegler e Dittrich (2007)](https://doi.org/10.1007/978-3-540-72677-7_3); [W3C PROV-O](https://www.w3.org/TR/prov-o/); [Bonduel et al. (2018)](https://ceur-ws.org/Vol-2159/04paper.pdf).

**Aplicação no projeto:** Perfil próprio: modelos, resultados e ocorrências de propriedades são registros informacionais derivados de prov:Entity.

**Limite:** Não implementa gUFO, gOCED ou GoDa. A classificação do registro não afirma identidade com o produto físico.


In [7]:
PERFIL_TTL = """
@prefix ctx: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#> .
@prefix int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#> .
@prefix civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ctx: a owl:Ontology ;
  rdfs:label "Perfil de organização e propriedades contextualizadas CIV2155"@pt ;
  rdfs:comment "Extensão própria, separada dos cinco grafos do N2. Distingue registros informacionais dos recursos físicos descritos por BOT. Não implementa gOCED/gUFO nem cria equivalências entre modelos."@pt ;
  rdfs:seeAlso <https://www.w3.org/TR/prov-o/>,
    <https://standards.buildingsmart.org/IFC/RELEASE/IFC4/FINAL/HTML/schema/ifcpropertyresource/lexical/ifcpropertysinglevalue.htm> .

ctx:RegistroInformacional a owl:Class ; rdfs:subClassOf prov:Entity ;
  rdfs:label "Registro informacional"@pt ;
  rdfs:comment "Recurso que registra uma representação, resultado ou evidência do estudo. Sua classificação não transforma a entidade descrita em registro nem afirma identidade entre ambos."@pt .
int:ModeloIFC rdfs:subClassOf ctx:RegistroInformacional ; rdfs:label "Modelo IFC de origem"@pt .
int:RelacaoEspacial rdfs:subClassOf ctx:RegistroInformacional ; rdfs:label "Evidência de relação espacial"@pt .
civ:ResultadoPorAmbiente rdfs:subClassOf ctx:RegistroInformacional ; rdfs:label "Resultado por ambiente"@pt .
ctx:ConjuntoPropriedadesIFC a owl:Class ; rdfs:subClassOf ctx:RegistroInformacional ; rdfs:label "Conjunto de propriedades IFC"@pt .
ctx:OcorrenciaPropriedadeIFC a owl:Class ; rdfs:subClassOf ctx:RegistroInformacional ;
  rdfs:label "Propriedade em seu conjunto de origem"@pt ;
  rdfs:comment "Registro contextual identificado pelo arquivo, Pset e STEP da propriedade. Não representa um novo produto físico."@pt .
ctx:UnidadeIFC a owl:Class ; rdfs:subClassOf ctx:RegistroInformacional ; rdfs:label "Unidade declarada no IFC"@pt .

ctx:conjuntoPropriedades a owl:ObjectProperty ; rdfs:label "conjunto de propriedades"@pt ; rdfs:range ctx:ConjuntoPropriedadesIFC .
ctx:propriedade a owl:ObjectProperty ; rdfs:domain ctx:ConjuntoPropriedadesIFC ; rdfs:range ctx:OcorrenciaPropriedadeIFC .
ctx:conjuntoFonte a owl:ObjectProperty ; rdfs:domain ctx:OcorrenciaPropriedadeIFC ; rdfs:range ctx:ConjuntoPropriedadesIFC .
ctx:produto a owl:ObjectProperty ; rdfs:label "produto descrito"@pt .
ctx:unidade a owl:ObjectProperty ; rdfs:range ctx:UnidadeIFC .
ctx:stepIFC a owl:DatatypeProperty ; rdfs:range xsd:integer .
ctx:globalIdPset a owl:DatatypeProperty ; rdfs:range xsd:string .
ctx:tipoIFC a owl:DatatypeProperty ; rdfs:range xsd:string .
ctx:tipoValorIFC a owl:DatatypeProperty ; rdfs:range xsd:string .
ctx:origemUnidade a owl:DatatypeProperty ; rdfs:range xsd:string .
ctx:tipoUnidade a owl:DatatypeProperty ; rdfs:range xsd:string .
ctx:nomeUnidade a owl:DatatypeProperty ; rdfs:range xsd:string .
ctx:prefixoUnidade a owl:DatatypeProperty ; rdfs:range xsd:string .
ctx:estadoLeitura a owl:DatatypeProperty ; rdfs:range xsd:string .
"""


### 3.2. Acrescentar o perfil e o esquema de referência

**O que este trecho faz:** Carrega o perfil próprio e o BOT 0.3.2 em grafos separados e mostra suas hierarquias.

**Referências deste trecho:** [Rasmussen et al. (2021), §3.4](https://doi.org/10.3233/SW-200385); [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [W3C PROV-O](https://www.w3.org/TR/prov-o/).

**Aplicação no projeto:** Preserva os cinco grafos de entrada. A licença e as anotações do esquema BOT são mantidas no arquivo integral.

**Limite:** Não executa raciocinador global. Esquemas e indivíduos são examinados separadamente.


In [8]:
dataset.graph(GRAPHS["perfil"]).parse(data=PERFIL_TTL, format="turtle")
dataset.graph(GRAPHS["bot-referencia"]).parse(ESQUEMA_BOT, format="turtle")
for chave in ["perfil", "bot-referencia"]:
    graph = dataset.graph(GRAPHS[chave])
    print(chave, len(graph), "triplas")
    for filho, pai in sorted(graph.subject_objects(RDFS.subClassOf), key=lambda t: str(t[0])):
        print(filho, "→", pai)


perfil 56 triplas
https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#ResultadoPorAmbiente → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#ConjuntoPropriedadesIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#OcorrenciaPropriedadeIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional → http://www.w3.org/ns/prov#Entity
https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#UnidadeIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#ModeloIFC → https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#RegistroInformacional
https://github.com/prcgui/bim-o

## 4. Recuperar o contexto das propriedades

### 4.1. Conjuntos, valores e unidades da V137

**O que este trecho faz:** Percorre Psets diretamente associados à V137, conserva nomes, números STEP, tipos nominais, valores e unidades.

**Referências deste trecho:** [Bonduel et al. (2018)](https://ceur-ws.org/Vol-2159/04paper.pdf); [IFC4, IfcPropertySingleValue](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/FINAL/HTML/schema/ifcpropertyresource/lexical/ifcpropertysinglevalue.htm); [IfcOpenShell, unidades](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/unit/index.html).

**Aplicação no projeto:** POC própria inspirada na distinção L1, L2 e L3. A unidade explícita prevalece; na ausência, get_property_unit procura a unidade do projeto para o tipo de medida.

**Limite:** Abrange somente valores simples diretamente associados. Não infere cm² ou cm³ a partir do centímetro linear, nem unidade para um IfcReal sem dimensão.


In [9]:
def contextualizar_v137(ds, identities, arquivo_ifc, hash_ifc):
    """POC delimitada: Psets diretamente associados, IfcPropertySingleValue.

    Mantém nomes originais, tipos nominais, valor e unidade nativa. IfcReal não
    recebe unidade por suposição. Não inclui propriedades herdadas de tipo,
    IfcElementQuantity nem outros subtipos de IfcProperty.
    """
    import ifcopenshell
    from ifcopenshell.util.unit import get_property_unit

    if sha(arquivo_ifc) != hash_ifc:
        raise ValueError("IFC SCO diferente da fonte integrada.")
    model = ifcopenshell.open(arquivo_ifc)
    entity = model.by_guid(V137)
    row = next(r for r in identities if r["modelo"] == "sco" and r["guid"] == V137)
    product = URIRef(row["uri"])
    graph = ds.graph(GRAPHS["propriedades-contextuais"])
    ns = Namespace(BASE + "id/contexto/" + hash_ifc[:12] + "/")
    processed = []
    unsupported = []
    for rel in sorted(entity.IsDefinedBy, key=lambda r: r.id()):
        if not rel.is_a("IfcRelDefinesByProperties"):
            continue
        pset = rel.RelatingPropertyDefinition
        if not pset.is_a("IfcPropertySet"):
            unsupported.append({"step": pset.id(), "tipo": pset.is_a()})
            continue
        pn = ns["pset/" + pset.GlobalId]
        for p, o in [
            (RDF.type, CTX.ConjuntoPropriedadesIFC),
            (RDFS.label, Literal(pset.Name or "")),
            (CTX.globalIdPset, Literal(pset.GlobalId)),
            (CTX.stepIFC, Literal(pset.id())),
            (CTX.produto, product),
            (PROV.wasDerivedFrom, B["modelo/sco"]),
        ]:
            graph.add((pn, p, o))
        graph.add((product, CTX.conjuntoPropriedades, pn))
        for prop in pset.HasProperties:
            if not prop.is_a("IfcPropertySingleValue") or prop.NominalValue is None:
                unsupported.append({"step": prop.id(), "tipo": prop.is_a()})
                continue
            node = ns["propriedade/" + pset.GlobalId + "/" + str(prop.id())]
            typ = prop.NominalValue.is_a()
            value = prop.NominalValue.wrappedValue
            # The nominal IFC type remains explicit even when RDF scalar datatypes coincide.
            literal = Literal(value)
            unit = get_property_unit(prop, model)
            origin = (
                "explicita"
                if prop.Unit is not None
                else (
                    "unidades_do_projeto_por_tipo_da_medida"
                    if unit
                    else "nao_declarada_ou_nao_dimensional"
                )
            )
            graph.add((pn, CTX.propriedade, node))
            for p, o in [
                (RDF.type, CTX.OcorrenciaPropriedadeIFC),
                (RDFS.label, Literal(prop.Name)),
                (CTX.stepIFC, Literal(prop.id())),
                (CTX.tipoValorIFC, Literal(typ)),
                (RDF.value, literal),
                (CTX.conjuntoFonte, pn),
                (CTX.origemUnidade, Literal(origin)),
                (PROV.wasDerivedFrom, B["modelo/sco"]),
            ]:
                graph.add((node, p, o))
            unit_info = None
            if unit:
                un = ns["unidade/" + str(unit.id())]
                graph.add((node, CTX.unidade, un))
                unit_info = {
                    "step": unit.id(),
                    "tipo": unit.is_a(),
                    "tipo_unidade": str(getattr(unit, "UnitType", "")),
                    "nome": str(getattr(unit, "Name", "")),
                    "prefixo": str(getattr(unit, "Prefix", None) or ""),
                }
                for p, o in [
                    (RDF.type, CTX.UnidadeIFC),
                    (CTX.stepIFC, Literal(unit.id())),
                    (CTX.tipoIFC, Literal(unit_info["tipo"])),
                    (CTX.tipoUnidade, Literal(unit_info["tipo_unidade"])),
                    (CTX.nomeUnidade, Literal(unit_info["nome"])),
                    (CTX.prefixoUnidade, Literal(unit_info["prefixo"])),
                    (PROV.wasDerivedFrom, B["modelo/sco"]),
                ]:
                    graph.add((un, p, o))
            processed.append(
                {
                    "uri": str(node),
                    "pset": pset.Name,
                    "pset_guid": pset.GlobalId,
                    "nome": prop.Name,
                    "step": prop.id(),
                    "tipo_valor_ifc": typ,
                    "valor": value,
                    "valor_lexical_rdf": str(literal),
                    "datatype": str(literal.datatype or XSD.string),
                    "unidade": unit_info,
                    "origem_unidade": origin,
                }
            )
    return {
        "produto_uri": str(product),
        "guid": V137,
        "nome": entity.Name,
        "step": entity.id(),
        "sha256_ifc": hash_ifc,
        "conjuntos": len(set(r["pset_guid"] for r in processed)),
        "propriedades": len(processed),
        "itens": processed,
        "nao_tratados": unsupported,
        "alcance": "Somente V137; Psets diretamente associados e valores simples; unidades nativas, sem harmonização ou cálculo estrutural.",
    }


### 4.2. Executar a contextualização

**O que este trecho faz:** Extrai o SCO sem alterar seus bytes, acrescenta a POC e apresenta as quatro quantidades em seus conjuntos.

**Referências deste trecho:** [IFC4, IfcPropertySingleValue](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/FINAL/HTML/schema/ifcpropertyresource/lexical/ifcpropertysinglevalue.htm); [IfcOpenShell, unidades](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/unit/index.html); [Bonduel et al. (2018)](https://ceur-ws.org/Vol-2159/04paper.pdf).

**Aplicação no projeto:** Resultado de referência: nove conjuntos e 34 propriedades simples, conservando unidades nativas.

**Limite:** O resultado não harmoniza globalmente todos os produtos ou propriedades dos três modelos.


In [10]:
ifc = TRABALHO / CONFIG_SCO["arquivo"]
with zipfile.ZipFile(localizar_entrada(CONFIG_SCO["zip"])) as z:
    ifc.write_bytes(z.read(CONFIG_SCO["arquivo"]))
poc = contextualizar_v137(dataset, identidades, ifc, CONFIG_SCO["sha256_ifc"])
salvar(SAIDA / "Propriedades_V137.json", poc)
dataset.graph(GRAPHS["propriedades-contextuais"]).serialize(
    SAIDA / "Propriedades_V137.nt", format="nt"
)
print(poc["conjuntos"], "conjuntos;", poc["propriedades"], "propriedades simples")
diretas = list(
    dataset.graph(GRAPHS["sco"]).objects(URIRef(v137["uri"]), P.quantidade_property_simple)
)
print("Valores diretos preservados:", [x.n3() for x in diretas])
for item in poc["itens"]:
    if item["nome"].startswith("Quantidade"):
        print(
            item["pset"], item["nome"], item["valor"], item["tipo_valor_ifc"], item["unidade"]
        )


9 conjuntos; 34 propriedades simples
Valores diretos preservados: ['"4.26151"^^<http://www.w3.org/2001/XMLSchema#decimal>', '"0.140544"^^<http://www.w3.org/2001/XMLSchema#decimal>', '"1.87392"^^<http://www.w3.org/2001/XMLSchema#decimal>', '"2.04999"^^<http://www.w3.org/2001/XMLSchema#decimal>']
AltoQi_Eberick-Item_01 Quantidade_01 0.140544 IfcVolumeMeasure {'step': 3, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'VOLUMEUNIT', 'nome': 'CUBIC_METRE', 'prefixo': ''}
AltoQi_Eberick-Item_02 Quantidade_02 1.87392 IfcAreaMeasure {'step': 4, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'AREAUNIT', 'nome': 'SQUARE_METRE', 'prefixo': ''}
AltoQi_Eberick-Item_03 Quantidade_03 4.26151 IfcMassMeasure {'step': 10, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'MASSUNIT', 'nome': 'GRAM', 'prefixo': 'KILO'}
AltoQi_Eberick-Item_04 Quantidade_04 2.04999 IfcMassMeasure {'step': 10, 'tipo': 'IfcSIUnit', 'tipo_unidade': 'MASSUNIT', 'nome': 'GRAM', 'prefixo': 'KILO'}


nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None


## 5. Validar um contrato explícito

### 5.1. Identidade, modelos e relações

**O que este trecho faz:** Expõe as restrições de GUID, origem, modelo e medidas dos vínculos espaciais.

**Referências deste trecho:** [W3C SHACL (2017)](https://www.w3.org/TR/shacl/); [Rasmussen et al. (2021), §3.4](https://doi.org/10.3233/SW-200385); [W3C PROV-O](https://www.w3.org/TR/prov-o/).

**Aplicação no projeto:** Restrições próprias do estudo, escritas na linguagem W3C SHACL. O contrato é construído pelos quatro trechos visíveis.

**Limite:** Conformidade com este contrato não certifica projeto estrutural, alinhamento dos IFCs, completude ou consistência OWL global.


In [11]:
RESTRICOES_SHACL_TTL = """
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix ctx: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#> .
@prefix int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#> .
@prefix civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#> .
@prefix b: <https://github.com/prcgui/bim-ontology-CIV2155/id/integracao/> .
@prefix bot: <https://w3id.org/bot#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix props: <http://lbd.arch.rwth-aachen.de/props#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# Contrato próprio do estudo, baseado na linguagem W3C SHACL (2017).
# Executado na projeção explícita documentada no Notebook 3. Sem inferência.
# ctx:ProdutoIndexado é apenas alvo local dessa projeção de validação.
ctx:ProdutoShape a sh:NodeShape ; sh:targetClass ctx:ProdutoIndexado ;
 sh:nodeKind sh:IRI ;
 sh:property [ sh:path props:globalIdIfcRoot_attribute_simple ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:string ; sh:minLength 22 ; sh:maxLength 22 ] ;
 sh:property [ sh:path prov:wasDerivedFrom ; sh:minCount 1 ; sh:maxCount 1 ; sh:class int:ModeloIFC ; sh:nodeKind sh:IRI ] .

ctx:ModeloShape a sh:NodeShape ; sh:targetClass int:ModeloIFC ;
 sh:property [ sh:path int:chave ; sh:minCount 1 ; sh:maxCount 1 ; sh:in ("arquitetura" "smt" "sco") ] ;
 sh:property [ sh:path int:grafo ; sh:minCount 1 ; sh:maxCount 1 ; sh:nodeKind sh:IRI ] ;
 sh:property [ sh:path int:sha256IFC ; sh:minCount 1 ; sh:maxCount 1 ; sh:pattern "^[0-9a-f]{64}$" ] .

ctx:RelacaoShape a sh:NodeShape ; sh:targetClass int:RelacaoEspacial ;
 sh:property [ sh:path int:ambiente ; sh:minCount 1 ; sh:maxCount 1 ; sh:class bot:Space ] ;
 sh:property [ sh:path int:elementoEstrutural ; sh:minCount 1 ; sh:maxCount 1 ; sh:class ctx:ProdutoIndexado ] ;
 sh:property [ sh:path int:modelo ; sh:minCount 1 ; sh:maxCount 1 ; sh:class int:ModeloIFC ] ;
 sh:property [ sh:path int:predicado ; sh:minCount 1 ; sh:maxCount 1 ; sh:in (bot:containsElement bot:intersectingElement bot:adjacentElement) ] ;
 sh:property [ sh:path int:estado ; sh:minCount 1 ; sh:maxCount 1 ; sh:in ("contido" "intersecciona" "adjacente") ] ;
 sh:property [ sh:path int:volumeIntersecaoM3 ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:double ; sh:minInclusive 0 ] ;
 sh:property [ sh:path int:volumeElementoM3 ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:double ; sh:minExclusive 0 ] ;
 sh:property [ sh:path int:areaContatoM2 ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:double ; sh:minInclusive 0 ] ;
 sh:sparql [ sh:message "Estado incompatível com o predicado BOT registrado." ; sh:select \"\"\"
 PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
 PREFIX bot: <https://w3id.org/bot#>
 SELECT $this WHERE { $this int:estado ?estado ; int:predicado ?p .
 FILTER((?estado = "contido" && ?p != bot:containsElement) ||
 (?estado = "intersecciona" && ?p != bot:intersectingElement) ||
 (?estado = "adjacente" && ?p != bot:adjacentElement)) }
 \"\"\" ] .

"""


### 5.2. Resultados de ambientes

**O que este trecho faz:** Expõe o contrato dos estados e das quantidades, incluindo a ausência de contagem em espaços inconclusivos.

**Referências deste trecho:** [W3C SHACL (2017)](https://www.w3.org/TR/shacl/).

**Aplicação no projeto:** Restrições próprias do estudo, escritas na linguagem W3C SHACL. O contrato é construído pelos quatro trechos visíveis.

**Limite:** Conformidade com este contrato não certifica projeto estrutural, alinhamento dos IFCs, completude ou consistência OWL global.


In [12]:
RESTRICOES_SHACL_TTL += """
ctx:ResultadoShape a sh:NodeShape ; sh:targetClass civ:ResultadoPorAmbiente ;
 sh:property [ sh:path civ:ambiente ; sh:minCount 1 ; sh:maxCount 1 ; sh:class bot:Space ] ;
 sh:property [ sh:path civ:estado ; sh:minCount 1 ; sh:maxCount 1 ; sh:in ("associacao_executada" "associacao_executada_com_pendencias" "inconclusivo_preparacao_espaco") ] ;
 sh:property [ sh:path civ:quantidadeEsquadrias ; sh:maxCount 1 ; sh:datatype xsd:integer ; sh:minInclusive 0 ] ;
 sh:sparql [ sh:message "Ambiente inconclusivo não pode receber contagem artificial." ; sh:select \"\"\"
 PREFIX civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#>
 SELECT $this WHERE { $this civ:estado "inconclusivo_preparacao_espaco" ; civ:quantidadeEsquadrias ?n }
 \"\"\" ] ;
 sh:sparql [ sh:message "Resultado executado deve declarar a contagem de esquadrias." ; sh:select \"\"\"
 PREFIX civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#>
 SELECT $this WHERE { $this civ:estado ?estado . FILTER(?estado != "inconclusivo_preparacao_espaco")
 FILTER NOT EXISTS { $this civ:quantidadeEsquadrias ?n } }
 \"\"\" ] .

"""


### 5.3. Contexto e unidade das propriedades

**O que este trecho faz:** Expõe restrições dos conjuntos, ocorrências e compatibilidade entre tipo nominal IFC e unidade.

**Referências deste trecho:** [W3C SHACL (2017)](https://www.w3.org/TR/shacl/); [IFC4, IfcPropertySingleValue](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/FINAL/HTML/schema/ifcpropertyresource/lexical/ifcpropertysinglevalue.htm); [IfcOpenShell, unidades](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/unit/index.html).

**Aplicação no projeto:** Restrições próprias do estudo, escritas na linguagem W3C SHACL. O contrato é construído pelos quatro trechos visíveis.

**Limite:** Conformidade com este contrato não certifica projeto estrutural, alinhamento dos IFCs, completude ou consistência OWL global.


In [13]:
RESTRICOES_SHACL_TTL += """
ctx:PsetShape a sh:NodeShape ; sh:targetClass ctx:ConjuntoPropriedadesIFC ;
 sh:property [ sh:path ctx:produto ; sh:minCount 1 ; sh:maxCount 1 ; sh:class ctx:ProdutoIndexado ] ;
 sh:property [ sh:path ctx:globalIdPset ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:string ] ;
 sh:property [ sh:path ctx:propriedade ; sh:minCount 1 ; sh:class ctx:OcorrenciaPropriedadeIFC ] .
ctx:PropriedadeShape a sh:NodeShape ; sh:targetClass ctx:OcorrenciaPropriedadeIFC ;
 sh:property [ sh:path ctx:conjuntoFonte ; sh:minCount 1 ; sh:maxCount 1 ; sh:class ctx:ConjuntoPropriedadesIFC ] ;
 sh:property [ sh:path rdfs:label ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:string ] ;
 sh:property [ sh:path ctx:stepIFC ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:integer ; sh:minExclusive 0 ] ;
 sh:property [ sh:path ctx:tipoValorIFC ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:string ] ;
 sh:property [ sh:path rdf:value ; sh:minCount 1 ; sh:maxCount 1 ] ;
 sh:property [ sh:path ctx:unidade ; sh:maxCount 1 ; sh:class ctx:UnidadeIFC ] ;
 sh:sparql [ sh:message "Medida dimensional selecionada deve indicar unidade recuperada do IFC." ; sh:select \"\"\"
 PREFIX ctx: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#>
 SELECT $this WHERE { $this ctx:tipoValorIFC ?t .
 FILTER(?t IN ("IfcLengthMeasure", "IfcPositiveLengthMeasure", "IfcAreaMeasure", "IfcVolumeMeasure", "IfcMassMeasure"))
 FILTER NOT EXISTS { $this ctx:unidade ?u } }
 \"\"\" ] ;
 sh:sparql [ sh:message "Tipo de unidade incompatível com o tipo nominal da medida IFC." ; sh:select \"\"\"
 PREFIX ctx: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#>
 SELECT $this WHERE { $this ctx:tipoValorIFC ?t ; ctx:unidade ?u . ?u ctx:tipoUnidade ?tipo .
 FILTER((?t = "IfcAreaMeasure" && ?tipo != "AREAUNIT") ||
 (?t = "IfcVolumeMeasure" && ?tipo != "VOLUMEUNIT") ||
 (?t = "IfcMassMeasure" && ?tipo != "MASSUNIT") ||
 (?t IN ("IfcLengthMeasure", "IfcPositiveLengthMeasure") && ?tipo != "LENGTHUNIT")) }
 \"\"\" ] .

"""


### 5.4. Coerência entre registros

**O que este trecho faz:** Expõe as consultas SHACL de correspondência entre vínculo BOT, registro espacial, modelo e GUID.

**Referências deste trecho:** [W3C SHACL (2017)](https://www.w3.org/TR/shacl/); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [W3C PROV-O](https://www.w3.org/TR/prov-o/).

**Aplicação no projeto:** Restrições próprias do estudo, escritas na linguagem W3C SHACL. O contrato é construído pelos quatro trechos visíveis.

**Limite:** Conformidade com este contrato não certifica projeto estrutural, alinhamento dos IFCs, completude ou consistência OWL global.


In [14]:
RESTRICOES_SHACL_TTL += """
# As verificações entre registros executam uma vez no estudo, sem produto cartesiano por alvo.
ctx:CoerenciaShape a sh:NodeShape ; sh:targetNode b:estudo ;
 sh:sparql [ sh:message "Registro aponta para relação BOT ausente." ; sh:select \"\"\"
 PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
 SELECT $this ?r WHERE { ?r a int:RelacaoEspacial ; int:ambiente ?a ; int:elementoEstrutural ?e ; int:predicado ?p .
 FILTER NOT EXISTS { ?a ?p ?e } }
 \"\"\" ] ;
 sh:sparql [ sh:message "Modelo da relação difere da proveniência do elemento." ; sh:select \"\"\"
 PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
 PREFIX prov: <http://www.w3.org/ns/prov#>
 SELECT $this ?r WHERE { ?r a int:RelacaoEspacial ; int:modelo ?m ; int:elementoEstrutural ?e .
 FILTER NOT EXISTS { ?e prov:wasDerivedFrom ?m } }
 \"\"\" ] ;
 sh:sparql [ sh:message "GUID repetido em produtos distintos do mesmo modelo." ; sh:select \"\"\"
 PREFIX props: <http://lbd.arch.rwth-aachen.de/props#>
 PREFIX prov: <http://www.w3.org/ns/prov#>
 SELECT $this ?e WHERE {
 ?e props:globalIdIfcRoot_attribute_simple ?guid ; prov:wasDerivedFrom ?m .
 ?outro props:globalIdIfcRoot_attribute_simple ?guid ; prov:wasDerivedFrom ?m .
 FILTER(?e != ?outro) }
 \"\"\" ] .
"""


### 5.5. Projetar os dados que o contrato examina

**O que este trecho faz:** Reúne os campos exigidos de produtos, modelos, relações, resultados e propriedades, preservando todos os alvos definidos.

**Referências deste trecho:** [W3C SHACL (2017)](https://www.w3.org/TR/shacl/); [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset).

**Aplicação no projeto:** ctx:ProdutoIndexado é um alvo local dessa projeção, sem modificar os grafos originais.

**Limite:** A projeção é declarada e não testa toda a semântica IFC ou todas as propriedades multivaloradas.


In [15]:
def projecao_validacao(ds, identities):
    """Projeção verificável dos campos do contrato; não valida toda a semântica IFC/OWL."""
    g = Graph()
    for row in identities:
        s = URIRef(row["uri"])
        g.add((s, RDF.type, CTX.ProdutoIndexado))
        for p in [RDF.type, P.globalIdIfcRoot_attribute_simple, PROV.wasDerivedFrom]:
            for o in ds.objects(s, p):
                g.add((s, p, o))
    for typ in [
        N.ModeloIFC,
        N.RelacaoEspacial,
        C.ResultadoPorAmbiente,
        CTX.ConjuntoPropriedadesIFC,
        CTX.OcorrenciaPropriedadeIFC,
        CTX.UnidadeIFC,
    ]:
        for s in ds.subjects(RDF.type, typ):
            for t in ds.triples((s, None, None)):
                g.add(t)
    for predicate in [BOT.containsElement, BOT.intersectingElement, BOT.adjacentElement]:
        for t in ds.triples((None, predicate, None)):
            g.add(t)
    for t in ds.triples((B.estudo, None, None)):
        g.add(t)
    return g


### 5.6. Definir e executar a validação

**O que este trecho faz:** Chama pySHACL sobre a projeção, grava o relatório e mostra a cobertura dos alvos.

**Referências deste trecho:** [W3C SHACL (2017)](https://www.w3.org/TR/shacl/); [pySHACL](https://github.com/RDFLib/pySHACL).

**Aplicação no projeto:** Inferência desativada. O relatório é produzido pelo contrato visível, com interrupção em caso de não conformidade.

**Limite:** Os ensaios de defeitos injetados permanecem no Drive; não integram os resultados primários.


In [16]:
def validar(ds, identities, raiz, saida):
    from pyshacl import validate

    g = projecao_validacao(ds, identities)
    t = time.perf_counter()
    conform, report, report_text = validate(
        g,
        shacl_graph=Graph().parse(data=RESTRICOES_SHACL_TTL, format="turtle"),
        inference="none",
        advanced=False,
        inplace=False,
    )
    out = Path(saida)
    out.mkdir(parents=True, exist_ok=True)
    if not isinstance(report, Graph):
        raise ValueError("Falha na especificação SHACL: " + str(report))
    report.serialize(out / "Validacao_SHACL.ttl", format="turtle")
    summary = {
        "conforme": bool(conform),
        "inferencia": "none",
        "triplas_da_projecao": len(g),
        "duracao_s": round(time.perf_counter() - t, 3),
        "alvos": {
            str(typ): len(set(g.subjects(RDF.type, typ)))
            for typ in [
                CTX.ProdutoIndexado,
                N.ModeloIFC,
                N.RelacaoEspacial,
                C.ResultadoPorAmbiente,
                CTX.ConjuntoPropriedadesIFC,
                CTX.OcorrenciaPropriedadeIFC,
            ]
        },
        "alcance": "Contrato de identidade, proveniência, resultados, vínculos e POC de propriedades. Não certifica qualidade estrutural, alinhamento, completude IFC nem consistência OWL global.",
    }
    salvar(out / "Resumo_SHACL.json", summary)
    if not conform:
        raise ValueError(report_text[:8000])
    return summary


validacao = validar(dataset, identidades, RAIZ, SAIDA)
display(validacao)
assert validacao["conforme"]
assert validacao["alvos"][str(CTX.ProdutoIndexado)] == 26601
assert validacao["alvos"][str(N.RelacaoEspacial)] == 1652


{'conforme': True,
 'inferencia': 'none',
 'triplas_da_projecao': 200962,
 'duracao_s': 77.53,
 'alvos': {'https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#ProdutoIndexado': 26601,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#ModeloIFC': 3,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#RelacaoEspacial': 1652,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#ResultadoPorAmbiente': 41,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#ConjuntoPropriedadesIFC': 9,
  'https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#OcorrenciaPropriedadeIFC': 34},
 'alcance': 'Contrato de identidade, proveniência, resultados, vínculos e POC de propriedades. Não certifica qualidade estrutural, alinhamento, completude IFC nem consistência OWL global.'}

## 6. Formular e executar as perguntas

### 6.1. Produtos e origem

**O que este trecho faz:** Conta produtos identificados por modelo e preserva o grafo de origem.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [W3C PROV-O](https://www.w3.org/TR/prov-o/).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [17]:
CONSULTAS_EXPLORACAO = {}
CONSULTAS_EXPLORACAO[
    "01_origem_produtos"
] = """
PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
PREFIX prov: <http://www.w3.org/ns/prov#>
SELECT ?modelo ?grafo (COUNT(DISTINCT ?produto) AS ?produtos)
WHERE { ?m a int:ModeloIFC ; int:chave ?modelo ; int:grafo ?grafo .
        ?produto prov:wasDerivedFrom ?m .
        GRAPH ?grafo { ?produto <http://lbd.arch.rwth-aachen.de/props#globalIdIfcRoot_attribute_simple> ?guid } }
GROUP BY ?modelo ?grafo ORDER BY ?modelo
"""


### 6.2. Ambientes e relações

**O que este trecho faz:** Recupera os resultados dos ambientes e suas relações estruturais.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [Rasmussen et al. (2021), §3.4](https://doi.org/10.3233/SW-200385).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [18]:
CONSULTAS_EXPLORACAO[
    "02_ambientes_e_relacoes"
] = """
PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
PREFIX civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#>
SELECT ?ambiente ?nome ?estado ?esquadrias ?modelo ?relacoes
WHERE { ?resultado a civ:ResultadoPorAmbiente ; civ:ambiente ?ambiente ; civ:nome ?nome ; civ:estado ?estado .
 OPTIONAL { ?resultado civ:quantidadeEsquadrias ?esquadrias }
 OPTIONAL { SELECT ?ambiente ?modelo (COUNT(?r) AS ?relacoes) WHERE {
 ?r a int:RelacaoEspacial ; int:ambiente ?ambiente ; int:modelo ?m . ?m int:chave ?modelo
 } GROUP BY ?ambiente ?modelo }
} ORDER BY ?nome ?modelo
"""


### 6.3. V137 e cozinha

**O que este trecho faz:** Busca a evidência do vínculo da V137 com a cozinha.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [Rasmussen et al. (2021), §3.4](https://doi.org/10.3233/SW-200385).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [19]:
CONSULTAS_EXPLORACAO[
    "03_cozinha_v137"
] = """
PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
PREFIX civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#>
PREFIX props: <http://lbd.arch.rwth-aachen.de/props#>
SELECT ?ambiente ?elemento ?predicado ?estado ?areaM2 ?volumeM3 ?modelo ?grafo ?sha256
WHERE { ?resultado civ:nome "COZINHA" ; civ:ambiente ?ambiente .
 ?elemento props:globalIdIfcRoot_attribute_simple "0EVSuIeKOHxuDHo7zKqQBn" .
 GRAPH <https://github.com/prcgui/bim-ontology-CIV2155/grafo/relacoes> {
 ?r a int:RelacaoEspacial ; int:ambiente ?ambiente ; int:elementoEstrutural ?elemento ;
 int:predicado ?predicado ; int:estado ?estado ; int:areaContatoM2 ?areaM2 ; int:volumeIntersecaoM3 ?volumeM3 ; int:modelo ?m }
 ?m int:chave ?modelo ; int:grafo ?grafo ; int:sha256IFC ?sha256 . }
"""


### 6.4. Ambientes inconclusivos

**O que este trecho faz:** Mostra os três estados sem contagem artificial.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [20]:
CONSULTAS_EXPLORACAO[
    "04_ambientes_inconclusivos"
] = """
PREFIX civ: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/analise#>
SELECT ?ambiente ?nome ?motivo ?quantidade
WHERE { ?r a civ:ResultadoPorAmbiente ; civ:ambiente ?ambiente ; civ:nome ?nome ;
 civ:estado "inconclusivo_preparacao_espaco" .
 OPTIONAL { ?r civ:motivo ?motivo } OPTIONAL { ?r civ:quantidadeEsquadrias ?quantidade } }
ORDER BY ?nome
"""


### 6.5. Propriedades contextualizadas

**O que este trecho faz:** Recupera Pset, propriedade, valor, tipo e unidade da POC.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [IFC4, IfcPropertySingleValue](https://standards.buildingsmart.org/IFC/RELEASE/IFC4/FINAL/HTML/schema/ifcpropertyresource/lexical/ifcpropertysinglevalue.htm).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [21]:
CONSULTAS_EXPLORACAO[
    "05_v137_propriedades_contextuais"
] = """
PREFIX ctx: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/contexto#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pset ?nome ?tipo ?valor ?unidade ?prefixo ?origemUnidade ?step
WHERE { ?conjunto a ctx:ConjuntoPropriedadesIFC ; rdfs:label ?pset ; ctx:propriedade ?p .
 ?p rdfs:label ?nome ; ctx:tipoValorIFC ?tipo ; rdf:value ?valor ; ctx:origemUnidade ?origemUnidade ; ctx:stepIFC ?step .
 OPTIONAL { ?p ctx:unidade ?u . ?u ctx:nomeUnidade ?unidade ; ctx:prefixoUnidade ?prefixo } }
ORDER BY ?pset ?nome
"""


### 6.6. GUID repetido no modelo

**O que este trecho faz:** Procura identidades repetidas dentro da mesma origem.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [W3C PROV-O](https://www.w3.org/TR/prov-o/).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [22]:
CONSULTAS_EXPLORACAO[
    "06_guids_repetidos_por_modelo"
] = """
PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX props: <http://lbd.arch.rwth-aachen.de/props#>
SELECT ?modelo ?guid (COUNT(DISTINCT ?produto) AS ?repeticoes)
WHERE { ?m a int:ModeloIFC ; int:chave ?modelo . ?produto prov:wasDerivedFrom ?m ; props:globalIdIfcRoot_attribute_simple ?guid }
GROUP BY ?modelo ?guid HAVING(COUNT(DISTINCT ?produto)>1)
"""


### 6.7. Armaduras fora da checagem

**O que este trecho faz:** Conta os produtos sem registro de avaliação geométrica no recorte.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [23]:
CONSULTAS_EXPLORACAO[
    "07_armaduras_sem_avaliacao_geometrica"
] = """
PREFIX beo: <https://pi.pauwel.be/voc/buildingelement#>
PREFIX int: <https://github.com/prcgui/bim-ontology-CIV2155/vocab/integracao#>
PREFIX prov: <http://www.w3.org/ns/prov#>
SELECT (COUNT(DISTINCT ?elemento) AS ?armaduras_sem_registro_espacial)
WHERE { GRAPH <https://github.com/prcgui/bim-ontology-CIV2155/grafo/sco> { ?elemento a beo:ReinforcingBar }
 ?elemento prov:wasDerivedFrom <https://github.com/prcgui/bim-ontology-CIV2155/id/integracao/modelo/sco> .
 FILTER NOT EXISTS { GRAPH <https://github.com/prcgui/bim-ontology-CIV2155/grafo/relacoes> { ?r int:elementoEstrutural ?elemento } } }
"""


### 6.8. Classes e níveis

**O que este trecho faz:** Examina as hierarquias dos esquemas sem confundir classes com produtos.

**Referências deste trecho:** [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197); [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [Rasmussen et al. (2021), §3.4](https://doi.org/10.3233/SW-200385).

**Aplicação no projeto:** Consulta própria em SPARQL 1.1. A orientação a perguntas dialoga com Liu e Yoon; não reproduz GoDa, NLP ou correspondência automática.

**Limite:** A resposta descreve o Dataset disponível. Ausência de um registro não demonstra ausência de uma relação física.


In [24]:
CONSULTAS_EXPLORACAO[
    "08_classes_e_niveis"
] = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?grafo ?classe ?superclasse
WHERE { GRAPH ?grafo { ?classe rdfs:subClassOf ?superclasse }
 FILTER(?grafo IN (<https://github.com/prcgui/bim-ontology-CIV2155/grafo/perfil>, <https://github.com/prcgui/bim-ontology-CIV2155/grafo/bot-referencia>)) }
ORDER BY ?grafo ?classe
"""


### 6.9. Executar as oito consultas e conferir as respostas

**O que este trecho faz:** Grava as consultas e respostas em JSON e CSV e confere casos selecionados de referência.

**Referências deste trecho:** [W3C SPARQL 1.1](https://www.w3.org/TR/sparql11-query/); [RDFLib 7.1.4](https://rdflib.readthedocs.io/en/7.1.4/); [Liu e Yoon (2024), §4](https://doi.org/10.1016/j.dss.2024.114197).

**Aplicação no projeto:** As consultas usadas são as oito definições anteriores, sem arquivos .rq externos.

**Limite:** As verificações conferem as perguntas do estudo. Não medem usabilidade nem qualidade global da integração.


In [25]:
def consultar(ds, raiz, saida):
    """Consultas publicadas e executáveis, com resultados efetivos em JSON/CSV."""
    results = []
    out = Path(saida)
    for nome, text in sorted(CONSULTAS_EXPLORACAO.items()):
        print("CONSULTA", nome, flush=True)
        result = ds.query(text)
        fields = [str(v) for v in result.vars]
        rows = [
            {k: str(v) if v is not None else None for k, v in zip(fields, row)}
            for row in result
        ]
        with (out / (nome + ".csv")).open("w", encoding="utf-8-sig", newline="") as fp:
            w = csv.DictWriter(fp, fieldnames=fields)
            w.writeheader()
            w.writerows(rows)
        results.append(
            {
                "id": nome,
                "consulta": text,
                "linhas": len(rows),
                "colunas": fields,
                "resultados": rows,
            }
        )
    salvar(out / "Consultas_Executadas.json", results)
    return results


consultas = consultar(dataset, RAIZ, SAIDA)
for consulta in consultas:
    print(consulta["id"], consulta["linhas"], "linhas")
display(consultas[2]["resultados"])
display(consultas[3]["resultados"])
display(consultas[4]["resultados"])
assert consultas[0]["linhas"] == 3
assert consultas[2]["linhas"] == 1
assert consultas[3]["linhas"] == 3
assert consultas[4]["linhas"] == poc["propriedades"]
assert consultas[5]["linhas"] == 0
assert consultas[6]["resultados"][0]["armaduras_sem_registro_espacial"] == "17755"


CONSULTA 01_origem_produtos


CONSULTA 02_ambientes_e_relacoes


CONSULTA 03_cozinha_v137


CONSULTA 04_ambientes_inconclusivos


CONSULTA 05_v137_propriedades_contextuais


CONSULTA 06_guids_repetidos_por_modelo


... (2 outputs left out)

01_origem_produtos 3 linhas
02_ambientes_e_relacoes 54 linhas
03_cozinha_v137 1 linhas
04_ambientes_inconclusivos 3 linhas
05_v137_propriedades_contextuais 34 linhas
06_guids_repetidos_por_modelo 0 linhas
07_armaduras_sem_avaliacao_geometrica 1 linhas
08_classes_e_niveis 11 linhas


[{'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_de3ed29d-12e1-45f6-af99-d3219f5ba16f',
  'elemento': 'https://github.com/prcgui/bim-ontology-CIV2155/id/estrutura-sco/beam_0e7dce12-a146-11ef-8351-c87f54d1a2f1',
  'predicado': 'https://w3id.org/bot#adjacentElement',
  'estado': 'adjacente',
  'areaM2': '0.4674837228519026',
  'volumeM3': '0.0',
  'modelo': 'sco',
  'grafo': 'https://github.com/prcgui/bim-ontology-CIV2155/grafo/sco',
  'sha256': 'bde759756938510c56112b9544aa2a26aca08b6a229008de85c5144d2961ea93'}]

[{'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_5e17dff5-4488-4291-abba-92395059a4d4',
  'nome': 'AMAMENTAÇÃO',
  'motivo': 'Ambiente não forma uma única casca fechada; sem reparação automática; recuperação controlada sem candidato aceito',
  'quantidade': None},
 {'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_bbfa3934-1b3d-4529-a9c4-10714c19a7a4',
  'nome': 'INFANTIL 1',
  'motivo': 'Ambiente não forma uma única casca fechada; sem reparação automática; recuperação controlada sem candidato aceito',
  'quantidade': None},
 {'ambiente': 'https://github.com/prcgui/bim-ontology-CIV2155/id/notebook1-v03/space_8600f390-0e41-48e4-b329-913a579a8477',
  'nome': 'INFANTIL 5',
  'motivo': 'Ambiente não forma uma única casca fechada; sem reparação automática; recuperação controlada sem candidato aceito',
  'quantidade': None}]

[{'pset': 'AltoQi_Eberick-Item_01',
  'nome': 'Descrição_01',
  'tipo': 'IfcText',
  'valor': 'Concreto - C-30 - Abatimento 12 cm',
  'unidade': None,
  'prefixo': None,
  'origemUnidade': 'nao_declarada_ou_nao_dimensional',
  'step': '430382'},
 {'pset': 'AltoQi_Eberick-Item_01',
  'nome': 'Quantidade_01',
  'tipo': 'IfcVolumeMeasure',
  'valor': '0.140544',
  'unidade': 'CUBIC_METRE',
  'prefixo': '',
  'origemUnidade': 'unidades_do_projeto_por_tipo_da_medida',
  'step': '430459'},
 {'pset': 'AltoQi_Eberick-Item_01',
  'nome': 'Status_01',
  'tipo': 'IfcText',
  'valor': 'Dimensionado',
  'unidade': None,
  'prefixo': None,
  'origemUnidade': 'nao_declarada_ou_nao_dimensional',
  'step': '432812'},
 {'pset': 'AltoQi_Eberick-Item_02',
  'nome': 'Descrição_02',
  'tipo': 'IfcText',
  'valor': 'Forma - Estrutura - Concreto',
  'unidade': None,
  'prefixo': None,
  'origemUnidade': 'nao_declarada_ou_nao_dimensional',
  'step': '430387'},
 {'pset': 'AltoQi_Eberick-Item_02',
  'nome': 'Qua

## 7. Gerar a saída para exploração

### 7.1. Selecionar termos e relações para o índice

**O que este trecho faz:** Define o índice de produtos, ambientes, contexto e esquemas, conservando tipos de termos RDF e grafos.

**Referências deste trecho:** [Shneiderman (1996), §§3 e 4](https://www.cs.umd.edu/~ben/papers/Shneiderman1996eyes.pdf); [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [Ziegler e Dittrich (2007)](https://doi.org/10.1007/978-3-540-72677-7_3).

**Aplicação no projeto:** Adapta visão geral, filtros, detalhes, relações e extração. O arquivo exploracao.json é gravado em resultados/organizacao e pode alimentar o explorador.

**Limite:** O índice não substitui o Dataset integral. Literais de geometria com mais de quatro mil caracteres e recursos auxiliares ficam no N-Quads. Não há avaliação de usabilidade com participantes.


In [26]:
def criar_indice(ds, identities, raiz, saida, poc, queries):
    """Índice de consulta visual, não substituto nem cópia integral do Dataset.

    Inclui os produtos e suas triplas diretas (exceto literais >4k caracteres),
    os registros N1/N2 e a extensão de propriedades. Termos RDF ficam tipados.
    Valores geométricos extensos continuam no Dataset integral disponibilizado.
    """
    terms = []
    lookup = {}
    quads = []
    omitted = Counter()

    def term(t):
        if isinstance(t, URIRef):
            v = ("u", str(t))
        elif isinstance(t, BNode):
            v = ("b", str(t))
        else:
            v = ("l", str(t), str(t.datatype or ""), str(t.language or ""))
        if v not in lookup:
            lookup[v] = len(terms)
            terms.append(v)
        return lookup[v]

    selected = {URIRef(r["uri"]) for r in identities}
    for typ in [
        N.ModeloIFC,
        N.RelacaoEspacial,
        C.ResultadoPorAmbiente,
        CTX.ConjuntoPropriedadesIFC,
        CTX.OcorrenciaPropriedadeIFC,
        CTX.UnidadeIFC,
    ]:
        selected.update(ds.subjects(RDF.type, typ))
    for s, p, o, c in ds.quads():
        if s not in selected:
            continue
        if isinstance(o, Literal) and len(str(o)) > 4000:
            omitted[str(p)] += 1
            continue
        quads.append([term(s), term(p), term(o), term(c)])
    products = [{**r, "t": term(URIRef(r["uri"]))} for r in identities]
    rooms = []
    spatial = list(
        csv.DictReader(
            (Path(raiz) / "resultados/integracao/Integracao_Por_Ambiente.csv").open(
                encoding="utf-8-sig"
            )
        )
    )
    spatial = {r["ambiente"]: r for r in spatial}
    for r in identities:
        if r["classe_ifc"] != "IfcSpace" or r["modelo"] != "arquitetura":
            continue
        s = URIRef(r["uri"])
        rec = next(ds.subjects(C.ambiente, s))
        status = str(ds.value(rec, C.estado))
        rooms.append(
            {
                "t": term(s),
                "guid": r["guid"],
                "nome": str(ds.value(rec, C.nome)),
                "estado_n1": status,
                "esquadrias": (
                    int(ds.value(rec, C.quantidadeEsquadrias))
                    if ds.value(rec, C.quantidadeEsquadrias) is not None
                    else None
                ),
                "estrutura": spatial[r["guid"]],
            }
        )
    # Full schema is separately inspectable, with its named-graph source.
    schema = []
    for key in ["perfil", "bot-referencia"]:
        for s, p, o in ds.graph(GRAPHS[key]):
            if p in [RDF.type, RDFS.label, RDFS.subClassOf, RDFS.domain, RDFS.range]:
                schema.append([term(s), term(p), term(o), term(GRAPHS[key])])
    data = {
        "versao": 1,
        "produtos": products,
        "ambientes": rooms,
        "termos": terms,
        "quadruplas": quads,
        "esquema": schema,
        "poc": poc,
        "consultas": queries,
        "resumo": {
            "produtos": len(products),
            "ambientes": len(rooms),
            "quadruplas_no_indice": len(quads),
            "literais_extensos_omitidos_por_predicado": dict(omitted),
            "sha256_n2": EXPECTED_SOURCE,
            "escopo": "Triplas diretas dos produtos e registros selecionados. Geometria extensa e recursos auxiliares permanecem no Dataset integral.",
        },
    }
    target = Path(saida) / "exploracao.json"
    salvar(target, data, True)
    salvar(
        Path(saida) / "Resumo_Indice.json", {**data["resumo"], "bytes": target.stat().st_size}
    )
    return data["resumo"]


### 7.2. Preservar os grafos e registrar a execução

**O que este trecho faz:** Confere os cinco grafos do N2, registra a atividade PROV-O e reabre o N-Quads para comparar os nove grafos.

**Referências deste trecho:** [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [W3C N-Quads](https://www.w3.org/TR/n-quads/); [W3C PROV-O](https://www.w3.org/TR/prov-o/).

**Aplicação no projeto:** A assinatura do código é calculada das oito funções visíveis, do perfil, das restrições e das consultas. O hash de entrada é o ZIP recebido nesta execução.

**Limite:** A atividade representa execução computacional. Não inventa eventos reais de construção. O arquivo recebido do N2 deve continuar inalterado.


In [27]:
def finalizar(ds, signatures, raiz, saida, inicio, resumo):
    """Confere os cinco grafos originais antes e depois da gravação N-Quads."""
    root = Path(raiz)
    out = Path(saida)
    for k, digest in signatures.items():
        if str(to_isomorphic(ds.graph(GRAPHS[k])).graph_digest()) != digest:
            raise ValueError("Grafo original alterado: " + k)
    run = URIRef(BASE + "id/execucao/n3/" + inicio.replace(":", "").replace("+", "_"))
    g = ds.graph(GRAPHS["execucao-n3"])
    for p, o in [
        (RDF.type, PROV.Activity),
        (PROV.startedAtTime, Literal(inicio, datatype=XSD.dateTime)),
        (
            PROV.endedAtTime,
            Literal(datetime.now(timezone.utc).isoformat(), datatype=XSD.dateTime),
        ),
        (PROV.used, URIRef(BASE + "id/entrada/n2/" + EXPECTED_SOURCE)),
        (CTX.sha256Entrada, Literal(EXPECTED_SOURCE)),
        (CTX.sha256Codigo, Literal(ASSINATURA_ORGANIZACAO)),
    ]:
        g.add((run, p, o))
    for graph in ["perfil", "bot-referencia", "propriedades-contextuais"]:
        g.add((GRAPHS[graph], PROV.wasGeneratedBy, run))
    raw = out / "Dataset_Organizado.nq"
    ds.serialize(raw, format="nquads")
    reopened = Dataset(default_union=True)
    reopened.parse(raw, format="nquads")
    for k in GRAPHS:
        if str(to_isomorphic(ds.graph(GRAPHS[k])).graph_digest()) != str(
            to_isomorphic(reopened.graph(GRAPHS[k])).graph_digest()
        ):
            raise ValueError("Serialização alterou termos do grafo " + k)
    with zipfile.ZipFile(out / "Dataset_Organizado.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(raw, raw.name)
    raw.unlink()
    summary = {
        **resumo,
        "grafos": {k: len(ds.graph(v)) for k, v in GRAPHS.items()},
        "quadruplas": sum(len(ds.graph(v)) for v in GRAPHS.values()),
        "cinco_grafos_n2_inalterados": True,
        "reabertura_sem_alteracao_de_termos": True,
        "sha256_entrada_n2": EXPECTED_SOURCE,
        "sha256_saida": sha(out / "Dataset_Organizado.zip"),
        "inicio_utc": inicio,
        "fim_utc": datetime.now(timezone.utc).isoformat(),
    }
    if (
        sha(root / "resultados/integracao/Integracao_Arquitetura_Estrutura.zip")
        != EXPECTED_SOURCE
    ):
        raise ValueError("Arquivo original foi alterado.")
    salvar(out / "Resumo_Organizacao.json", summary)
    return summary


### 7.3. Executar a geração do índice e do Dataset organizado

**O que este trecho faz:** Chama as funções anteriores, mostra o índice e o resumo e confere a preservação.

**Referências deste trecho:** [W3C RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset); [W3C PROV-O](https://www.w3.org/TR/prov-o/); [Shneiderman (1996), §§3 e 4](https://www.cs.umd.edu/~ben/papers/Shneiderman1996eyes.pdf).

**Aplicação no projeto:** Resultado de referência: 416.603 quádruplas em nove grafos; os cinco grafos do N2 devem permanecer idênticos.

**Limite:** A execução atual tem seus próprios tempos e sua proveniência. A organização não aumenta por si só a acurácia geométrica.


In [28]:
import inspect

ASSINATURA_ORGANIZACAO = hashlib.sha256(
    (
        "\n".join(
            inspect.getsource(f)
            for f in [
                salvar,
                carregar,
                contextualizar_v137,
                projecao_validacao,
                validar,
                consultar,
                criar_indice,
                finalizar,
            ]
        )
        + PERFIL_TTL
        + RESTRICOES_SHACL_TTL
        + json.dumps(CONSULTAS_EXPLORACAO, sort_keys=True)
    ).encode()
).hexdigest()
indice = criar_indice(dataset, identidades, RAIZ, SAIDA, poc, consultas)
resumo = finalizar(
    dataset,
    assinaturas_n2,
    RAIZ,
    SAIDA,
    INICIO_UTC,
    {
        "produtos": len(identidades),
        "psets_poc": poc["conjuntos"],
        "propriedades_poc": poc["propriedades"],
        "relacoes_n2": 1652,
        "pares_n1": 101,
        "shacl_conforme": validacao["conforme"],
    },
)
display(indice)
display(resumo)
assert resumo["cinco_grafos_n2_inalterados"] and resumo["reabertura_sem_alteracao_de_termos"]


{'produtos': 26601,
 'ambientes': 41,
 'quadruplas_no_indice': 394067,
 'literais_extensos_omitidos_por_predicado': {},
 'sha256_n2': '1df6f71ae28bc7726d70226b3cbb2eb8df2538b298acd5cacf409a3d379e094b',
 'escopo': 'Triplas diretas dos produtos e registros selecionados. Geometria extensa e recursos auxiliares permanecem no Dataset integral.'}

{'produtos': 26601,
 'psets_poc': 9,
 'propriedades_poc': 34,
 'relacoes_n2': 1652,
 'pares_n1': 101,
 'shacl_conforme': True,
 'grafos': {'arquitetura': 80234,
  'smt': 84933,
  'sco': 205194,
  'relacoes': 18194,
  'proveniencia': 26620,
  'perfil': 56,
  'bot-referencia': 957,
  'propriedades-contextuais': 406,
  'execucao-n3': 9},
 'quadruplas': 416603,
 'cinco_grafos_n2_inalterados': True,
 'reabertura_sem_alteracao_de_termos': True,
 'sha256_entrada_n2': '1df6f71ae28bc7726d70226b3cbb2eb8df2538b298acd5cacf409a3d379e094b',
 'sha256_saida': '79ddb7602461c0452aa6363429c44770df548b89f39d4be6f8ea17cd5e2086a3',
 'inicio_utc': '2026-11-03T22:23:36.451813+00:00',
 'fim_utc': '2026-11-03T22:29:12.539289+00:00'}

## Reunir os resultados desta execução

**O que este trecho faz:** Cria um ZIP na pasta oficial de resultados e oferece o download. O arquivo não inclui a si próprio.

**Referências deste trecho:** [Python 3.12](https://docs.python.org/3.12/library/); [Google Colab e Drive](https://research.google.com/colaboratory/faq.html).

**Aplicação no projeto:** Empacotamento próprio; os resultados já permanecem no Drive quando a etapa inicial foi executada no Colab.

**Limite:** Uma sessão antiga que ainda aponta para /content precisa copiar seus arquivos ao Drive antes de encerrar.


In [29]:
PACOTE_RESULTADOS = SAIDA / "Resultados_Notebook3.zip"
with zipfile.ZipFile(PACOTE_RESULTADOS, "w", zipfile.ZIP_DEFLATED) as pacote:
    for arquivo in sorted(SAIDA.rglob("*")):
        if arquivo.is_file() and arquivo.resolve() != PACOTE_RESULTADOS.resolve():
            pacote.write(arquivo, arquivo.relative_to(SAIDA).as_posix())
print("Pacote disponível:", caminho_no_projeto(PACOTE_RESULTADOS))
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(PACOTE_RESULTADOS))


Pacote disponível: resultados/organizacao/Resultados_Notebook3.zip


## Interpretação e referências

O ganho demonstrado é preservar origem e contexto, distinguir registros e produtos, recuperar propriedades da V137 com suas unidades, executar oito perguntas e verificar um contrato explícito. Não foram demonstrados alinhamento externo dos IFCs, identidade física entre modelos, adequação estrutural, harmonização global, aumento de acurácia geométrica ou ganho de usabilidade medido com participantes.

A [bibliografia aplicada](https://github.com/prcgui/bim-ontology-CIV2155/blob/main/referencias/organizacao/README.md) discrimina as fontes e os limites. O [explorador publicado](https://prcgui.github.io/bim-ontology-CIV2155/) usa um índice estático; este notebook gera um novo índice e o Dataset completo na pasta de resultados. Auditorias, comparações e ensaios ficam no Drive.
